In [1]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Downloading the count data for the bicycles.

In [2]:
BASE_URL = "https://opendata.apps.mow.vlaanderen.be/fietstellingen"

COUNT_COLS  = ['site ID', 'richting', 'type', 'van', 'tot', 'aantal']
COUNT_DTYPE = {
    'site ID':  'int32',
    'richting': 'category',
    'type':     'category',
    'van':      'string',
    'tot':      'string',
    'aantal':   'float32',
}

def download_counts(year: int, month: int, save_dir: str = "data") -> pd.DataFrame:
    Path(save_dir).mkdir(exist_ok=True)
    filename   = f"data-{year}-{month:02d}.csv"
    local_path = Path(save_dir) / filename

    if local_path.exists():
        print(f"  {filename} already cached locally")
    else:
        url = f"{BASE_URL}/{filename}"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        local_path.write_bytes(resp.content)
        print(f"  Downloaded {filename} ({len(resp.content)/1024:.0f} KB)")

    df = pd.read_csv(
        local_path,
        sep=',',
        header=None,
        names=COUNT_COLS,
        dtype=COUNT_DTYPE,
        low_memory=False,
    )
    df = df[df['type'] == 'FIETSERS']
    df['ds'] = pd.to_datetime(df['van'], errors='coerce').dt.normalize()
    daily = (
        df.groupby(['site ID', 'ds'], observed=True)['aantal']
          .sum()
          .reset_index()
    )
    return daily

MONTHS_TO_DOWNLOAD = (
    [(2024, m) for m in range(1, 13)]
    + [(2025, m) for m in range(1, 13)]
    + [(2026, m) for m in range(1, 5)]
)

print("Downloading cycling count data...")
daily_frames = []
for year, month in MONTHS_TO_DOWNLOAD:
    try:
        daily_frames.append(download_counts(year, month))
    except Exception as e:
        print(f"  Skipping {year}-{month:02d}: {e}")

raw_daily = pd.concat(daily_frames, ignore_index=True)
print(f"\nTotal daily rows: {len(raw_daily):,}")
print(raw_daily.head(3))

  data-2024-01.csv already cached locally
  data-2024-02.csv already cached locally
  data-2024-03.csv already cached locally
  data-2024-04.csv already cached locally
  data-2024-05.csv already cached locally
  data-2024-06.csv already cached locally
  data-2024-07.csv already cached locally
  data-2024-08.csv already cached locally
  data-2024-09.csv already cached locally
  data-2024-10.csv already cached locally
  data-2024-11.csv already cached locally
  data-2024-12.csv already cached locally
  data-2025-01.csv already cached locally
  data-2025-02.csv already cached locally
  data-2025-03.csv already cached locally
  data-2025-04.csv already cached locally
  data-2025-05.csv already cached locally
  data-2025-06.csv already cached locally
  data-2025-07.csv already cached locally
  data-2025-08.csv already cached locally
  data-2025-09.csv already cached locally
  data-2025-10.csv already cached locally
  data-2025-11.csv already cached locally
  data-2025-12.csv already cached 

## Loading the sites.csv for the lat/long calculation via the weather matching 

In [3]:
sites_url  = f"{BASE_URL}/sites.csv"
sites_cols = ['site_id','site_nr','long','lat','naam','domein',
              'wegnr','district','gemeente','interval','datum_van']

sites = pd.read_csv(sites_url, header=None, names=sites_cols)
sites['datum_van'] = pd.to_datetime(sites['datum_van'])
print(f"\nSites loaded: {len(sites)}")


Sites loaded: 151


In [4]:
daily = (
    raw_daily
    .groupby(['site ID', 'ds'], observed=True)['aantal']
    .sum()
    .reset_index()
    .rename(columns={'site ID': 'unique_id', 'aantal': 'y'})
)

def make_full_index(g):
    return pd.DataFrame({
        'unique_id': g['unique_id'].iloc[0],
        'ds': pd.date_range(g['ds'].min(), g['ds'].max(), freq='D')
    })

full_idx = pd.concat([make_full_index(g) for _, g in daily.groupby('unique_id')])
daily    = full_idx.merge(daily, on=['unique_id', 'ds'], how='left')

def fill_gaps(s):
    s = s.interpolate('linear', limit=3)
    mask = s.isna()
    s[mask] = s.shift(7).combine_first(s.shift(-7))[mask]
    return s

daily['y'] = daily.groupby('unique_id')['y'].transform(fill_gaps)

print(f"Daily dataframe shape: {daily.shape}")
print(daily.head(3))

Daily dataframe shape: (119691, 3)
   unique_id         ds      y
0          1 2024-01-01  107.0
1          1 2024-01-02  147.0
2          1 2024-01-03  166.0


## Outlier Detection (Rolling Median)

Two-layer fault detection. First pass: flag any count exceeding 4× the local 30-day median as suspect, NaN it, and re-run the gap-fill. The second pass (Isolation Forest) runs after the weather merge so it can use weather-context features.

In [5]:
rolling_med = daily.groupby('unique_id')['y'].transform(
    lambda x: x.rolling(30, min_periods=7, center=True).median()
)
suspect = (daily['y'] > 4 * rolling_med) & rolling_med.notna()
print(f"Rolling-median outliers flagged: {int(suspect.sum())}")
daily.loc[suspect, 'y'] = np.nan

daily['y'] = daily.groupby('unique_id')['y'].transform(fill_gaps)
print(f"Remaining NaN in y: {int(daily['y'].isna().sum())}")

Rolling-median outliers flagged: 190
Remaining NaN in y: 222


## Attach Weather Covariates

Weather is downloaded by the companion notebook and pushed to Hugging Face. Hourly readings are aggregated into the four daily features the model expects:

| Daily feature         | Source (hourly)      | Aggregation |
|---                    |---                   |---          |
| `temperature_2m_max`  | `temperature_2m`     | max         |
| `precipitation_sum`   | `precipitation`      | sum         |
| `wind_speed_10m_max`  | `wind_speed_10m`     | max         |
| `sunshine_duration`   | `sunshine_duration`  | sum (seconds) |

The weather `site_id` matches the cycling `unique_id`, so the join is direct.

In [6]:
from huggingface_hub import hf_hub_download

weather_hourly = pd.read_csv(hf_hub_download(
    repo_id="Zelal-iab/MDA_Project_Data",
    filename="weather_data_2024-2026.csv",
    repo_type="dataset",
))
weather_hourly['time'] = pd.to_datetime(weather_hourly['time'])
weather_hourly['ds']   = weather_hourly['time'].dt.normalize()

weather_daily = (
    weather_hourly
    .groupby(['site_id', 'ds'])
    .agg(
        temperature_2m_max = ('temperature_2m',    'max'),
        precipitation_sum  = ('precipitation',     'sum'),
        wind_speed_10m_max = ('wind_speed_10m',    'max'),
        sunshine_duration  = ('sunshine_duration', 'sum'),
    )
    .reset_index()
    .rename(columns={'site_id': 'unique_id'})
)

daily['unique_id']         = daily['unique_id'].astype('int64')
weather_daily['unique_id'] = weather_daily['unique_id'].astype('int64')

daily = daily.merge(weather_daily, on=['unique_id', 'ds'], how='left')
coverage = 1 - daily['temperature_2m_max'].isna().mean()
print(f"Daily shape: {daily.shape}")
print(f"Weather coverage: {coverage * 100:.1f}%")
print(daily.head(3))

Daily shape: (119691, 7)
Weather coverage: 96.4%
   unique_id         ds      y  temperature_2m_max  precipitation_sum  \
0          1 2024-01-01  107.0                8.90                2.0   
1          1 2024-01-02  147.0               12.10               21.9   
2          1 2024-01-03  166.0               10.25                3.6   

   wind_speed_10m_max  sunshine_duration  
0           10.104455       21616.066704  
1           11.815245           0.000000  
2           10.125710       14196.805530  


## Isolation Forest

Per-site multivariate anomaly detection on top of the rolling-median check. The Isolation Forest catches subtler patterns — for example a count that is only suspicious because it occurred during heavy rain on a Tuesday in January. Flagged days are kept in the dataset as a `sensor_fault` binary covariate (fed in as `hist_exog`), rather than dropped, so the model can learn to discount them.

In [7]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def build_iso_features(g):
    g = g.copy()
    g['dow']           = g['ds'].dt.dayofweek
    g['month']         = g['ds'].dt.month
    g['rolling_med']   = g['y'].rolling(30, min_periods=7, center=True).median()
    g['deviation']     = g['y'] - g['rolling_med']
    g['vs_last_week']  = g['y'] - g['y'].shift(7)
    g['vs_4weeks_ago'] = g['y'] - g['y'].shift(28)
    g['rain_spike']    = (g['precipitation_sum']   > 10).astype(int)
    g['sun_zero']      = (g['sunshine_duration'] == 0).astype(int)
    return g

iso_features = ['deviation', 'vs_last_week', 'vs_4weeks_ago',
                'dow', 'month', 'rain_spike', 'sun_zero']

flag_frames = []
for uid, group in daily.groupby('unique_id'):
    g = build_iso_features(group).dropna(subset=iso_features)
    if len(g) < 60:
        continue
    X = StandardScaler().fit_transform(g[iso_features].values)
    clf = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
    g['sensor_fault'] = (clf.fit_predict(X) == -1).astype(int)
    flag_frames.append(g[['unique_id', 'ds', 'sensor_fault']])

flags = pd.concat(flag_frames, ignore_index=True)
daily = daily.merge(flags, on=['unique_id', 'ds'], how='left')
daily['sensor_fault'] = daily['sensor_fault'].fillna(0).astype('int8')

print(f"Sites scored: {flags['unique_id'].nunique()}")
print(f"Sensor-fault flags: {int(daily['sensor_fault'].sum())} of {len(daily):,}")

Sites scored: 148
Sensor-fault flags: 3505 of 119,691


## Normalize

Sites range from a handful of cyclists per day to thousands. Per-series z-score on `y` so the global model weights each site equally during training. Weather variables get a single global scaler. Scalers are kept for inverse-transform at evaluation time.

In [8]:
from sklearn.preprocessing import StandardScaler

y_scalers = {}
for uid, grp in daily.groupby('unique_id'):
    sc  = StandardScaler()
    idx = daily['unique_id'] == uid
    daily.loc[idx, 'y'] = sc.fit_transform(grp[['y']].values).ravel()
    y_scalers[uid] = sc

weather_cols = ['temperature_2m_max', 'precipitation_sum',
                'wind_speed_10m_max', 'sunshine_duration']
daily[weather_cols] = daily[weather_cols].fillna(daily[weather_cols].median())

wx_scaler = StandardScaler()
daily[weather_cols] = wx_scaler.fit_transform(daily[weather_cols])

print(f"y  mean {daily['y'].mean():+.3f}  std {daily['y'].std():.3f}")
print(daily[['unique_id', 'ds', 'y'] + weather_cols + ['sensor_fault']].head(3))

y  mean -0.000  std 1.000
   unique_id         ds         y  temperature_2m_max  precipitation_sum  \
0          1 2024-01-01 -1.806489           -0.863476          -0.066919   
1          1 2024-01-02 -1.553226           -0.398725           4.888328   
2          1 2024-01-03 -1.432926           -0.667409           0.331493   

   wind_speed_10m_max  sunshine_duration  sensor_fault  
0            2.447444          -0.288130             0  
1            3.307473          -1.574349             0  
2            2.458129          -0.729597             0  


## Format for `neuralforecast` & Temporal Split

Drop sites with fewer than 365 valid days so the 1-year lookback window is satisfiable everywhere. Split temporally (never random for time series):

- **Train:** start of data → 2025-09-30
- **Val:**   2025-10-01 → 2025-12-31
- **Test:**  2026-01-01 → end of data

In [10]:
TRAIN_END = pd.Timestamp('2025-09-30')
VAL_END   = pd.Timestamp('2025-12-31')

valid_sites = (
    daily[daily['ds'] <= VAL_END]
    .dropna(subset=['y'])
    .groupby('unique_id')['y']
    .count()
    .loc[lambda x: x >= 457]
    .index
)
daily = (
    daily[daily['unique_id'].isin(valid_sites)]
    .dropna(subset=['y'])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)
print(f"Sites kept: {len(valid_sites)}  |  Rows: {len(daily):,}")


train_df = daily[daily['ds'] <= TRAIN_END].copy()
val_df   = daily[(daily['ds'] > TRAIN_END) & (daily['ds'] <= VAL_END)].copy()
test_df  = daily[daily['ds'] > VAL_END].copy()

print(f"Train: {train_df['ds'].min().date()} -> {train_df['ds'].max().date()}  ({len(train_df):,} rows)")
print(f"Val:   {val_df['ds'].min().date()} -> {val_df['ds'].max().date()}  ({len(val_df):,} rows)")
print(f"Test:  {test_df['ds'].min().date()} -> {test_df['ds'].max().date()}  ({len(test_df):,} rows)")

Sites kept: 137  |  Rows: 115,888
Train: 2024-01-01 -> 2025-09-30  (87,083 rows)
Val:   2025-10-01 -> 2025-12-31  (12,512 rows)
Test:  2026-01-01 -> 2026-04-30  (16,293 rows)


## N-BEATSx Model

Global model trained jointly across all sites with weather as historical and future exogenous variables. Interpretable mode (trend + seasonality stacks) so per-site trend slopes and seasonal amplitudes can be extracted directly from the fitted stacks.

In [ ]:
%pip install neuralforecast==2.11.0 lightning==2.5.0

In [17]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx

HIST_EXOG = ['temperature_2m_max', 'precipitation_sum',
             'wind_speed_10m_max', 'sunshine_duration', 'sensor_fault']
FUTR_EXOG = ['temperature_2m_max', 'precipitation_sum',
             'wind_speed_10m_max', 'sunshine_duration']

model = NBEATSx(
    h                         = 7,
    input_size                = 365,
    hist_exog_list            = HIST_EXOG,
    futr_exog_list            = FUTR_EXOG,
    stack_types               = ['trend', 'seasonality'],
    max_steps                 = 2000,
    early_stop_patience_steps = 10,
    val_check_steps           = 50,
    random_seed               = 42
)

fit_df   = pd.concat([train_df, val_df], ignore_index=True)
val_size = val_df['ds'].nunique()

nf = NeuralForecast(models=[model], freq='D')
nf.fit(df=fit_df, val_size=val_size)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  5.4 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 5.4 M                                                                                            
Non-trainable params: 5.6 K                                                                                        
Total params: 5.4 M                                                                                                
Total estimated model params size (MB): 21                                                                         
Modules in train mode: 22                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

In [18]:
forecast_start = VAL_END + pd.Timedelta(days=1)
forecast_end   = VAL_END + pd.Timedelta(days=7)

futr_df = nf.make_future_dataframe()

weather_future = (
    daily[(daily['ds'] >= forecast_start) & (daily['ds'] <= forecast_end)]
    [['unique_id', 'ds'] + FUTR_EXOG]
)
futr_df = futr_df.merge(weather_future, on=['unique_id', 'ds'], how='left')
futr_df[FUTR_EXOG] = futr_df[FUTR_EXOG].fillna(0)

forecasts = nf.predict(futr_df=futr_df).reset_index()
print(f"Forecast rows: {len(forecasts)}  |  sites: {forecasts['unique_id'].nunique()}")
print(forecasts.head())


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Forecast rows: 959  |  sites: 137
   index  unique_id         ds   NBEATSx
0      0          1 2026-01-01 -1.101782
1      1          1 2026-01-02 -1.362146
2      2          1 2026-01-03 -2.116841
3      3          1 2026-01-04 -1.559067
4      4          1 2026-01-05 -0.656720


## Evaluation

Inverse-transform predictions and observations back to raw cyclist counts via the per-site scalers, then compute accuracy metrics on the 7-day horizon immediately following the validation cutoff (first week of the test set).

Metrics reported:

- **MAE** – mean absolute error, in cyclists/day.
- **RMSE** – penalises large misses more heavily than MAE.
- **MAPE / sMAPE** – scale-free percentage errors; sMAPE is bounded and behaves better on low-count sites.
- **R²** – fraction of variance in the held-out counts explained by the model (1.0 is perfect, 0.0 matches predicting the mean).
- **Seasonal-naive baseline** – repeats the value from 7 days earlier; the gap between N-BEATSx and this baseline is the value added by the model.
- **Per-horizon-day MAE** – shows how accuracy decays from day +1 to day +7.

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

merged = forecasts.merge(
    test_df[['unique_id', 'ds', 'y']].rename(columns={'y': 'y_true'}),
    on=['unique_id', 'ds'],
    how='inner',
)

eval_frames = []
for uid, grp in merged.groupby('unique_id'):
    sc = y_scalers.get(uid)
    if sc is None:
        continue
    g = grp.copy()
    g['y_true']  = sc.inverse_transform(g[['y_true']].values).ravel()
    g['NBEATSx'] = sc.inverse_transform(g[['NBEATSx']].values).ravel()
    eval_frames.append(g)

eval_df = pd.concat(eval_frames, ignore_index=True)
y_true  = eval_df['y_true'].to_numpy()
y_pred  = eval_df['NBEATSx'].to_numpy()

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
nz   = y_true > 1e-6
mape = np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100 if nz.any() else float('nan')

print("7-day horizon performance (raw cyclist counts)")
print(f"  MAE   : {mae:.2f}")
print(f"  RMSE  : {rmse:.2f}")
print(f"  MAPE  : {mape:.2f}%")
print(f"  R^2   : {r2:.3f}")
print(f"  Sites : {eval_df['unique_id'].nunique()}  |  Rows: {len(eval_df):,}")
print(f"  Mean observed: {y_true.mean():.1f}  (MAE / mean = {mae / y_true.mean() * 100:.1f}%)")

7-day horizon performance (raw cyclist counts)
  MAE   : 117.01
  RMSE  : 196.00
  MAPE  : 173.93%
  R^2   : 0.693
  Sites : 136  |  Rows: 952
  Mean observed: 211.4  (MAE / mean = 55.3%)
